## Joins in MapReduce


It’s like when you have two lists (or tables) and you want to match them based on a common column (like an ID).

Reduce-Side Join: Combines data after sorting and sending it to different computers.

Map-Side Join: Works faster when one dataset is very small (like merging a small lookup table).

Semi-Join: Filters one table first, so less data moves between computers.



In [1]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MapReduceJoin").getOrCreate()
df1 = spark.createDataFrame([(1,"A"),(2,"B")], ["id","value1"])
df2 = spark.createDataFrame([(1,"X"),(2,"Y")], ["id","value2"])

joined = df1.join(df2, "id", "inner")
joined.show()


+---+------+------+
| id|value1|value2|
+---+------+------+
|  1|     A|     X|
|  2|     B|     Y|
+---+------+------+



## Sqoop
Sqoop helps move big data between databases (like MySQL, Oracle) and Hadoop (big data storage).

It can send data both ways:

Database → Hadoop

Hadoop → Database

It also changes the format of the data into Hadoop-friendly types like Parquet.

In [2]:
import pandas as pd
df = pd.DataFrame({"id":[1,2,3],"name":["A","B","C"]})
df.to_csv("mysql_export.csv", index=False)

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SqoopSim").getOrCreate()
data = spark.read.csv("mysql_export.csv", header=True, inferSchema=True)
data.show()


+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  3|   C|
+---+----+



## Interacting with Hive and Impala
Hive: Good for running big, slow batch jobs (like making monthly reports).

Impala: Runs fast, interactive queries (like checking today’s sales quickly).

Both let you use SQL (the language used for databases).

In [3]:
spark.sql("CREATE OR REPLACE TEMP VIEW sample AS SELECT 1 as id, 'A' as name")
spark.sql("SELECT * FROM sample").show()


+---+----+
| id|name|
+---+----+
|  1|   A|
+---+----+



## Working with Hive and Impala
Hive: Great for ETL (Extract, Transform, Load) jobs and handling complicated data.

Impala: Great for real-time analysis because it runs faster.

Both support partitioning (splitting data for speed).

In [4]:
data = spark.createDataFrame([(1,"HR"),(2,"Finance")], ["id","dept"])
data.write.partitionBy("dept").mode("overwrite").parquet("partitioned_data")
spark.read.parquet("partitioned_data").show()


+---+-------+
| id|   dept|
+---+-------+
|  1|     HR|
|  2|Finance|
+---+-------+



## Data Types in Hive
Primitive types: simple values like numbers, text, dates.

Complex types: structured values like lists, maps, nested data.

Picking the right type saves space and speeds up processing.

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("skills", ArrayType(StringType()), True)
])
df = spark.createDataFrame([("John", ["Python","SQL"])], schema)
df.show(truncate=False)


+----+-------------+
|name|skills       |
+----+-------------+
|John|[Python, SQL]|
+----+-------------+



## Validation of Data
Making sure the data is correct before using it:

Columns are correct type

Values are correct

Partitions (separated folders/tables) are correct

This prevents errors in analysis.

In [8]:
expected_schema = {"id": "int", "name": "string"}
actual_schema = {f.name: f.dataType.simpleString() for f in df.schema.fields}
print(actual_schema == expected_schema)


False


## HCatalog: Unified Data Management
Think of it like a phonebook for all your data.

It stores information (metadata) about tables and files so other Hadoop tools know where and how to use the data.

It helps different tools work together smoothly.

In [7]:
tables = spark.catalog.listTables()
print(tables)


[Table(name='sample', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
